# 01 — Bias-Corrected LLM-as-a-Judge Audit on RAGTruth

This notebook audits an LLM-as-a-judge evaluator against human hallucination annotations on `data/processed/ragtruth_features.parquet`.

## Audit Design
- **Stratified sampling** (`sample_size <= 300`, `random_state=42`) across `(task_type, has_hallucination)`.
- **Paired bias probes** for every sampled item:
  1. `base`: Context -> Prompt -> Response (`position=0`).
  2. `length_expanded`: Non-factual neutral framing added to Response (`position=0`) to isolate **verbosity bias**.
  3. `position_swapped`: Response -> Prompt -> Context (`position=1`) to isolate **position bias**.
- **Bias correction (`fit_bias_correction`)**: Ridge regression controlling for `human_label`, `response_length`, and `position`, then residualizing length and position offsets.
- **Honest TBD gating**: If no `DEEPSEEK_API_KEY` or `GROQ_API_KEY` is configured, `run_judge_audit` writes `status: "tbd"` without fabricating scores.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

ROOT = (Path("..") if Path("../src").exists() else Path(".")).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.eval.judge_audit import (
    compute_judge_metrics,
    fit_bias_correction,
    run_judge_audit,
)

FEATURES_PARQUET = ROOT / "data/processed/ragtruth_features.parquet"
ARTIFACT_JSON = ROOT / "artifacts" / "judge_audit_summary.json"

print(f"Features parquet: {FEATURES_PARQUET} (exists={FEATURES_PARQUET.exists()})")
print(f"Audit artifact:   {ARTIFACT_JSON} (exists={ARTIFACT_JSON.exists()})")

In [ ]:
if ARTIFACT_JSON.exists():
    audit_summary = json.loads(ARTIFACT_JSON.read_text())
else:
    audit_summary = run_judge_audit(FEATURES_PARQUET, ARTIFACT_JSON, sample_size=300)

print(f"Status: {audit_summary['status']}")
if audit_summary["status"] == "tbd":
    print(f"Reason: {audit_summary['reason']}")
    print("Real LLM judge scores are NOT available without an API key; no scores fabricated.")
else:
    print(f"Provider: {audit_summary.get('provider')} | Sample size: {audit_summary['sample_size']}")

## Agreement & Bias Visualization

The helper below renders comparison tables and lightweight SVG bar charts (with optional `matplotlib` if installed) for:
1. Per-task judge vs. human agreement (Cohen's kappa & Spearman $r$).
2. Raw vs. bias-corrected agreement and estimated verbosity / position coefficients.

In [ ]:
def render_bar_svg(title: str, items: list[tuple[str, float, str]], max_val: float = 1.0) -> HTML:
    rows = []
    for label, val, color in items:
        width = max(0.0, min(1.0, abs(val) / max_val)) * 260
        rows.append(
            f'<div style="display:flex;align-items:center;margin:4px 0;font-family:monospace;">'
            f'<span style="width:190px;">{label}</span>'
            f'<svg width="280" height="18"><rect x="0" y="2" width="{width:.1f}" height="14" '
            f'rx="3" fill="{color}"/></svg>'
            f'<span style="margin-left:8px;">{val:+.4f}</span></div>'
        )
    return HTML(f"<div style='margin:8px 0;'><strong>{title}</strong>{''.join(rows)}</div>")


def display_audit_report(metrics: dict, bias: dict, label_prefix: str = "") -> None:
    task_rows = []
    for task, stats in metrics["by_task"].items():
        task_rows.append(
            {
                "task_type": task,
                "n_samples": stats["n_samples"],
                "cohens_kappa": round(stats["cohens_kappa"], 4),
                "spearman_r": round(stats["spearman_r"], 4),
                "accuracy": round(stats["accuracy"], 4),
            }
        )
    display(pd.DataFrame(task_rows))

    display(
        render_bar_svg(
            f"{label_prefix}Agreement: Raw vs Bias-Corrected",
            [
                ("Raw Spearman r", bias["raw_spearman_r"], "#64748b"),
                ("Corrected Spearman r", bias["corrected_spearman_r"], "#0f766e"),
                ("Raw Cohen kappa", bias["raw_cohens_kappa"], "#94a3b8"),
                ("Corrected Cohen kappa", bias["corrected_cohens_kappa"], "#0284c7"),
            ],
            max_val=1.0,
        )
    )
    print(
        f"Estimated verbosity_coef={bias['verbosity_coef']:.6f} (per char), "
        f"position_coef={bias['position_coef']:.4f} (Response-first offset)"
    )

In [ ]:
if audit_summary["status"] == "ready":
    display_audit_report(
        audit_summary["metrics"],
        audit_summary["bias_correction"],
        label_prefix="[Live Audit] ",
    )
else:
    # Diagnostic demonstration only (clearly synthetic; never saved as live audit output)
    rng = np.random.default_rng(42)
    n = 120
    y = rng.integers(0, 2, size=n)
    lengths = rng.integers(40, 500, size=n)
    positions = rng.integers(0, 2, size=n)
    tasks = np.random.default_rng(42).choice(["QA", "Summary", "Data2txt"], size=n)
    biased_scores = np.clip(
        0.15 + 0.48 * y + 0.0009 * lengths + 0.14 * positions + rng.normal(0, 0.05, size=n),
        0.0,
        1.0,
    )
    diag_metrics = compute_judge_metrics(y, biased_scores, tasks)
    diag_bias = fit_bias_correction(biased_scores, lengths, positions, y)
    display_audit_report(diag_metrics, diag_bias, label_prefix="[Synthetic Diagnostic] ")